# RestoreAI — Temporary Colab GPU server

**Run cells in order.** Select **Runtime → Change runtime type → T4 GPU**. The Colab session must stay alive while the phone uses the server. This is a prototype, not a production deployment. Never upload private photos to GitHub. The first model download can take time.

In [ ]:
import torch, sys
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Choose a GPU runtime before continuing'

In [ ]:
import os, subprocess, pathlib, sys
from pathlib import Path
ROOT = Path('/content/RestoreAI')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','feature/bilingual-ui','https://github.com/khuloodakram/RestoreAI.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin','feature/bilingual-ui'],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','-B','feature/bilingual-ui','origin/feature/bilingual-ui'],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-r','server/requirements.txt','-r','server/requirements-lama.txt'],check=True)
# BasicSR 1.4.2 legacy packaging may fail on Python 3.13. Import from source instead.
BASICSR_ROOT = Path('/content/BasicSR')
if not BASICSR_ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','v1.4.2','https://github.com/XPixelGroup/BasicSR.git',str(BASICSR_ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','addict','future','lmdb','pyyaml','requests','scikit-image','scipy','tb-nightly','tqdm','yapf==0.40.1'],check=True)
os.environ['PYTHONPATH'] = str(BASICSR_ROOT) + os.pathsep + str(ROOT) + os.pathsep + os.environ.get('PYTHONPATH','')
sys.path.insert(0,str(BASICSR_ROOT))
print('BasicSR source prepared:',BASICSR_ROOT)


In [ ]:
from pathlib import Path
# Patch the checked-out BasicSR source before importing it.
p = BASICSR_ROOT / 'basicsr/data/degradations.py'
old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
new = 'from torchvision.transforms.functional import rgb_to_grayscale'
text = p.read_text()
if old in text:
    p.write_text(text.replace(old,new))
    print('Applied torchvision compatibility patch')
# Source checkout skips the packaging step that generates basicsr/version.py.
version_file = BASICSR_ROOT / 'basicsr/version.py'
if not version_file.exists():
    version_file.write_text("__version__ = '1.4.2'\\n__gitsha__ = 'unknown'\\n")
    print('Created BasicSR version metadata')
from basicsr.archs.rrdbnet_arch import RRDBNet
print('RRDBNet import OK')


In [ ]:
import urllib.request
weights = Path('/content/RealESRGAN_x4plus.pth')
url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
if not weights.exists():
    urllib.request.urlretrieve(url, weights)
assert weights.stat().st_size > 10_000_000, 'Checkpoint download seems incomplete'
os.environ['RESTOREAI_WEIGHTS'] = str(weights)
os.environ['RESTOREAI_RESTORE_ENGINE'] = 'lama'
import secrets
TOKEN = secrets.token_urlsafe(32)
os.environ['RESTOREAI_API_TOKEN'] = TOKEN
print('Model checkpoint ready. Token created (keep private).')

In [ ]:
import subprocess, time, requests, sys
server_log = open('/content/restoreai_uvicorn.log','w')
server = subprocess.Popen([sys.executable,'-m','uvicorn','server.app:app','--host','127.0.0.1','--port','8000'], cwd=str(ROOT),env=os.environ.copy(),stdout=server_log,stderr=subprocess.STDOUT)
for _ in range(35):
    if server.poll() is not None:
        raise RuntimeError(Path('/content/restoreai_uvicorn.log').read_text()[-4000:])
    try:
        status = requests.get('http://127.0.0.1:8000/health',timeout=2)
        if status.ok:
            print(status.json())
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('API did not start; inspect /content/restoreai_uvicorn.log')

In [ ]:
import urllib.request, subprocess, re, time, pathlib
binary = Path('/content/cloudflared')
if not binary.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',binary)
    binary.chmod(0o755)
log_path = Path('/content/cloudflared.log')
tunnel_log = open(log_path,'w')
tunnel = subprocess.Popen([str(binary),'tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'],stdout=tunnel_log,stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(45):
    if tunnel.poll() is not None:
        raise RuntimeError(log_path.read_text()[-3000:])
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_path.read_text())
    if matches:
        PUBLIC_URL = matches[-1]
        break
    time.sleep(2)
assert PUBLIC_URL, 'Tunnel did not become ready; inspect /content/cloudflared.log'
print('SERVER URL:', PUBLIC_URL)
print('ACCESS TOKEN:', TOKEN)
print('Enter these in RestoreAI → Connect temporary Colab server.')

## Verify before using a personal photo

The `/health` endpoint checks configuration, **not** actual LaMa or Real-ESRGAN inference. First test using a non-sensitive photo and a mask; large images may exhaust free GPU memory. The endpoint accepts at most **2 megapixels / 8 MB** and the full 4× output can be large. Do not share the access token or public URL. When finished, terminate the Colab runtime to stop the tunnel.

In [ ]:
from PIL import Image, ImageDraw
import io, requests
photo = Image.new('RGB',(128,128),(155,135,115))
d = ImageDraw.Draw(photo)
d.rectangle((15,15,110,110),fill=(90,130,155))
d.line((30,15,90,110),fill='white',width=4)
mask = Image.new('L',(128,128),0)
ImageDraw.Draw(mask).line((30,15,90,110),fill=255,width=7)
a,b = io.BytesIO(),io.BytesIO()
photo.save(a,'PNG');mask.save(b,'PNG')
response = requests.post('http://127.0.0.1:8000/enhance',
  headers={'Authorization':'Bearer '+TOKEN},
  files={'image':('sample.png',a.getvalue(),'image/png'),
         'mask':('mask.png',b.getvalue(),'image/png')},
  data={'mode':'restore'},timeout=300)
print('Inference HTTP:',response.status_code, 'bytes:',len(response.content))
if response.ok:
    Path('/content/restoreai_test_result.png').write_bytes(response.content)
    print('Real LaMa + Real-ESRGAN inference succeeded. Output: /content/restoreai_test_result.png')
else:
    print(response.text[:600]); print(Path('/content/restoreai_uvicorn.log').read_text()[-1500:])
    raise RuntimeError('Inference test failed')